# Getting started with MemsArrayWS objects

The `MemsArrayWS` class allows getting signals from a remote antenna running a local *Megamicros Broadcast Server (MBS)* server. 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from megamicros.log import log
from megamicros.core.ws import MemsArrayWS

log.setLevel( "INFO" )

# Set server access credentials
#HOST = 'buzenval20.fr'
#HOST = 'parisparc.biimea.tech
HOST = 'localhost'
PORT = 9002

## Connecting to the remote server

Providing a *MBS* server is running at ``HOST:PORT``, one can try to connect by creating a ``MemsArrayWS`` object.

In [2]:
# Define the antenna
try:
    antenna = MemsArrayWS( HOST, port=PORT )
except Exception as e:
    print( f"Failed: {e}" )


2023-11-03 17:24:39,246 [INFO]:  .Install MemsArrayWS settings
2023-11-03 17:24:39,248 [INFO]:  .Created a new antenna
2023-11-03 17:24:39,249 [INFO]:  .Async event loop already running. Adding coroutine to the event loop...


2023-11-03 17:24:39,253 [INFO]:  .Try connecting to ws://localhost:9002...
2023-11-03 17:24:39,273 [INFO]:  .Received positive answer from server
2023-11-03 17:24:39,273 [INFO]:  .Getting settings values from remote receiver...
2023-11-03 17:24:39,274 [INFO]:  .Received settings from server [ok]
2023-11-03 17:24:39,274 [INFO]:  .Set 32 available MEMs numbered from 0 to 31
2023-11-03 17:24:39,275 [INFO]:  .No analogic channels available
2023-11-03 17:24:39,276 [INFO]:  .Starting MegamicrosWS device [ready]


## Performing a selftest

In [ ]:
# Perform an antenna selftest
antenna.selftest()

2023-11-03 17:24:42,955 [INFO]:  .Connecting to remote host localhost:9002...
2023-11-03 17:24:42,957 [INFO]:  .Connected
2023-11-03 17:24:42,958 [INFO]:  .Send selftest command to server
2023-11-03 17:24:43,485 [INFO]:  .Remote server selftest command successfull


## Halting the remote server
Notice that the conection is lost. As such you could not restart the server.

In [ ]:
# Stop the remote server
antenna.shutdown()


## Running

### Getting signals from some MEMs

In [ ]:
# 2 seconds run, getting signals from MEMs 1 and 2
antenna.run(
    mems = [1, 2],
    duration=2,
    buffer_length=512,
    signal_q_size = 0,
)

# Init a np.ndarray
signals = np.ndarray( (0, antenna.channels_number ) )

# Get signals
i = 0
for data in antenna:
    i += 1
    signals = np.concatenate( ( signals, data ), axis=0 )

# waiting for the end of the running thread is mandatory
antenna.wait()
print( f"Exit from loop. Received {i} frames. Signal shape is: {np.shape( signals )}" )

## Plotting signals

In [ ]:
# plot signals
time = np.array( range( np.size(signals,0) ) )/antenna.sampling_frequency
fig, axs = plt.subplots( antenna.channels_number )
fig.suptitle('Channels activity')	
for s in range( antenna.channels_number ):
    axs[s].plot( time, signals[:,s] )
    axs[s].set( xlabel='time in seconds', ylabel='channel %d' % s )

plt.show()

## Saving signals as wav file
Since wavfiles are audio files, you cannot save more than 2 channels.

In [ ]:
import wave

WAV_FILENAME = 'toto.wav'

# 2 seconds run, getting signals from MEMs 1 and 2
antenna.run(
    mems = [1, 2],
    duration=10,
    buffer_length=512,
    signal_q_size = 0,
)

with  wave.open( WAV_FILENAME, mode='wb' ) as wavfile:
    wavfile.setnchannels(2)
    wavfile.setsampwidth(2)
    wavfile.setframerate( antenna.sampling_frequency )

    # Get signals
    for data in antenna:
        signal = data >> 4
        wavfile.writeframesraw( np.int16( np.reshape( signal, np.size( signal ), order='F' ) ) )

# waiting for the end of the running thread is mandatory
antenna.wait()

## Hearing signal with *pyaudio* library

Note that `signal_q_size` is set to 0, setting the internal queue to an infinite length.
This prevents breaks in the audio stream.

In [ ]:
import pyaudio

FRAME_LENGTH = 512
antenna.setSamplingFrequency( 50000 )

# Instantiate PyAudio and initialize PortAudio system resources (1)
p = pyaudio.PyAudio()

# Open stream
stream = p.open(
    format = pyaudio.paFloat32,
    channels = 2,
    rate = int( antenna.sampling_frequency ),
    output=True,
    frames_per_buffer=FRAME_LENGTH,
)

# Start running the remote Megamicros system
antenna.run( 
    mems=[3, 4],
    duration=10,
    frame_length=FRAME_LENGTH,
    counter_skip = True,
    signal_q_size = 0
)

# Get signals
transfers_counter = 0
for data in antenna:
    signal = data >> 4

    # convert into float and normalize with MEMs sensibility
    data = ( data.astype( np.float32 ).T * antenna.sensibility )

    # write into audio stream
    stream.write( data, num_frames=FRAME_LENGTH )
    transfers_counter += 1

# Close stream and release PortAudio system resources (5)
stream.close()            
p.terminate()

antenna.wait()


## Saving signals as H5 files

You can save signal in H5 file format. In this example sigansl are saved on the MBS remote server.
The antenna receive no more signals. 

In [ ]:
antenna.run(
    mems = [3, 4],
    duration=2,
    buffer_length=512,
    h5_recording=True,                          # H5 recording ON
    h5_pass_through=True,                       # perform F5 recording on server
    h5_rootdir='./',                            # directory where to save file
    h5_compressing=False,                       # Use compression or not
    background_mode=True,
    signal_q_size = 0,
)

antenna.wait()

## Getting signals yourself

In this example, signals are received using the antenna internal queue.

In [ ]:
import queue

antenna.run(
    mems = [1, 2],
    duration=2,
    buffer_length=512,
    signal_q_size = 0,
)

i = 0
while True:
    try:
        data = antenna.signal_q.get( timeout=5 )
        print( f"[{i}]" )
        i += 1
        # do what you want with data...

    except queue.Empty:
        print( f"exit from loop at i={i}" )
        break

antenna.wait()

## Listening to the Megamicro remote server
By starting a *master* run on the server, you can connect to the server from others hosts and listening to the signal stream.

### Staring the master run
This call lets the remote server starting a run in the background mode.

In [ ]:

antenna.run(
    mems = [1, 2, 3, 4],
    duration=0,
    buffer_length=512,
    signal_q_size=0,
    job='master', 
)

2023-11-03 17:55:08,426 [INFO]:  .Starting run execution
2023-11-03 17:55:08,428 [INFO]:  .Install MemsArray settings
2023-11-03 17:55:08,429 [INFO]:  .4 MEMs were activated among 0 to 31 available MEMs
2023-11-03 17:55:08,429 [INFO]:  .Install MemsArrayWS settings
2023-11-03 17:55:08,430 [INFO]:  .Pre-execution checks for MemsArray.run()
2023-11-03 17:55:08,430 [INFO]:  .Requested job: master
2023-11-03 17:55:08,431 [INFO]:  .Run infinite loop (duration=0)
2023-11-03 17:55:08,431 [INFO]:  .H5 recording off
2023-11-03 17:55:08,431 [INFO]:  .Start a `master` running job on remote server
2023-11-03 17:55:08,431 [INFO]:  .Background execution mode off
2023-11-03 17:55:08,432 [INFO]:  .Run thread execution started
2023-11-03 17:55:08,434 [INFO]:  .Connecting to remote host localhost:9002...


2023-11-03 17:55:08,437 [INFO]:  .Connected
2023-11-03 17:55:08,438 [INFO]:  .Send running job command (master)
2023-11-03 17:55:08,439 [INFO]:  .Master run command accepted by server
2023-11-03 17:55:10,444 [INFO]:  .Halt connection with server and exit


In [5]:
# Define the antenna
try:
    listener = MemsArrayWS( HOST, port=PORT )
except Exception as e:
    print( f"Failed: {e}" )


2023-11-03 17:25:12,905 [INFO]:  .Install MemsArrayWS settings
2023-11-03 17:25:12,906 [INFO]:  .Created a new antenna
2023-11-03 17:25:12,907 [INFO]:  .Async event loop already running. Adding coroutine to the event loop...


2023-11-03 17:25:12,909 [INFO]:  .Try connecting to ws://localhost:9002...
2023-11-03 17:25:12,911 [INFO]:  .Received positive answer from server
2023-11-03 17:25:12,912 [INFO]:  .Getting settings values from remote receiver...
2023-11-03 17:25:12,913 [INFO]:  .Received settings from server [ok]
2023-11-03 17:25:12,913 [INFO]:  .Set 32 available MEMs numbered from 0 to 31
2023-11-03 17:25:12,913 [INFO]:  .No analogic channels available
2023-11-03 17:25:12,915 [INFO]:  .Starting MegamicrosWS device [ready]


In [10]:
listener.setFrameLength(512)
listener.run(
    mems = [1, 2],
    buffer_length=512,
    signal_q_size=0,
    duration=2,
    job='listen'
)

# Init a np.ndarray
signals = np.ndarray( (0, listener.channels_number ) )

# Get signals
i = 0
for data in listener:
    i += 1
    signals = np.concatenate( ( signals, data ), axis=0 )
    print( data )

# waiting for the end of the running thread is mandatory
listener.wait()
print( f"Exit from loop. Received {i} frames. Signal shape is: {np.shape( signals )}" )

2023-11-03 17:56:15,977 [INFO]:  .Starting run execution
2023-11-03 17:56:15,979 [INFO]:  .Install MemsArray settings
2023-11-03 17:56:15,980 [INFO]:  .2 MEMs were activated among 0 to 31 available MEMs
2023-11-03 17:56:15,981 [INFO]:  .Install MemsArrayWS settings
2023-11-03 17:56:15,982 [INFO]:  .Pre-execution checks for MemsArray.run()
2023-11-03 17:56:15,983 [INFO]:  .Requested job: listen
2023-11-03 17:56:15,983 [INFO]:  .Perform a 2s run loop
2023-11-03 17:56:15,984 [INFO]:  .H5 recording off
2023-11-03 17:56:15,985 [INFO]:  .Start a `listen` running job on remote server
2023-11-03 17:56:15,986 [INFO]:  .Background execution mode off
2023-11-03 17:56:15,987 [INFO]:  .Run thread execution started
2023-11-03 17:56:15,988 [INFO]:  .Starting iterations: will produce data as numpy array of int32 (512 x 2 size)
2023-11-03 17:56:15,990 [INFO]:  .Connecting to remote host localhost:9002...
2023-11-03 17:56:15,993 [INFO]:  .Connected
2023-11-03 17:56:15,994 [INFO]:  .Send running job comm

[[ 471  192]
 [1994  268]
 [ 535  186]
 ...
 [ 129 1499]
 [ 374  268]
 [ 524  669]]
[[ 194  321]
 [1054 1743]
 [ 282  362]
 ...
 [1193 -258]
 [ 117 -663]
 [ 425 -370]]
[[ -109   -78]
 [ -391 -1275]
 [ -462    95]
 ...
 [-1122 -1328]
 [  199  -719]
 [-1163  -634]]
[[ -478    54]
 [-1322  2709]
 [ -295   792]
 ...
 [ 1588 -1063]
 [  644   106]
 [ -122 -1186]]
[[  308   -82]
 [ -804  -907]
 [ -204  -899]
 ...
 [ -851  1356]
 [ -573   880]
 [-1027  1332]]
[[ 323  214]
 [1443 -136]
 [ 441 1177]
 ...
 [  24 -570]
 [ 778 -579]
 [1159  275]]
[[-507 -127]
 [-824  453]
 [-465  254]
 ...
 [ 174  578]
 [ 332  107]
 [ -87  833]]
[[  71  209]
 [ 546  111]
 [ -32  549]
 ...
 [ 441  588]
 [ 591  193]
 [1081  626]]
[[  -38  -547]
 [  908 -1247]
 [ -122 -1226]
 ...
 [-1473  -314]
 [ -884    64]
 [-1209   433]]
[[-301 -581]
 [ 555 -489]
 [ 311 -339]
 ...
 [ 217 -612]
 [  33  109]
 [-417  -38]]
[[-482 -219]
 [-406 -102]
 [-128  145]
 ...
 [-403  737]
 [-374  -40]
 [-116 -673]]
[[ -347  -413]
 [  726 -1878

2023-11-03 17:56:17,992 [INFO]:  .Thread timer started for 2s duration
2023-11-03 17:56:18,012 [INFO]:  .Send stop command
2023-11-03 17:56:18,014 [INFO]:  .Received aknowledgment reponse from halt request. Stop running.


[[  405    90]
 [ 2082  -371]
 [  403  -484]
 ...
 [ -840  -773]
 [ -283  -466]
 [ -205 -1307]]
[[ -363  -459]
 [ -752 -2175]
 [ -452  -743]
 ...
 [-2345  1800]
 [ -877   892]
 [-1594   441]]
[[  277    34]
 [ 1776  -401]
 [  281  -338]
 ...
 [ -371   772]
 [ -205   222]
 [-1141  -625]]
[[   37  -138]
 [  682 -2815]
 [  263 -1166]
 ...
 [-2620  1488]
 [-1328    95]
 [-2948   798]]
[[ 108  135]
 [1414  435]
 [-237  326]
 ...
 [1033 3643]
 [ 552  823]
 [1154 2751]]
[[  667  -251]
 [ 3736 -1194]
 [  708  -115]
 ...
 [-1486 -3710]
 [ -465 -1787]
 [-1027 -3446]]
[[ -525  -336]
 [-3796 -3082]
 [-1967  -357]
 ...
 [-2835  1821]
 [ -966  1042]
 [-1217  2171]]
[[-155 -125]
 [2489 2355]
 [ 962 1426]
 ...
 [2289 1159]
 [1241  564]
 [2381 1380]]
[[  153  -524]
 [ 1161 -2253]
 [  541  -684]
 ...
 [-2218 -4300]
 [-1050 -1357]
 [-1877 -2004]]
[[ -740   305]
 [-3691  1054]
 [-1318   600]
 ...
 [ 1078 -1557]
 [  533  -730]
 [ 1521 -1038]]
[[  162  -196]
 [-1566  1177]
 [ -613   399]
 ...
 [  524  -104]

In [11]:
# halt job
antenna.halt()

2023-11-03 17:58:08,295 [INFO]:  .Connecting to remote host localhost:9002...
2023-11-03 17:58:08,299 [INFO]:  .Connected
2023-11-03 17:58:08,299 [INFO]:  .Send halt command...
2023-11-03 17:58:10,307 [INFO]:  .Halt command completed


In [12]:
# halt server
antenna.shutdown()

2023-11-03 18:03:27,392 [INFO]:  .Async event loop already running. Adding coroutine to the event loop...


2023-11-03 18:03:27,397 [INFO]:  .Connecting to remote host localhost:9002...
2023-11-03 18:03:27,400 [INFO]:  .Connected
2023-11-03 18:03:27,401 [INFO]:  .Send shutdown command to server
2023-11-03 18:03:27,402 [INFO]:  .Remote server shutdown success
